In [1]:
!nvidia-smi

Sat Apr 18 17:23:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
!pip install ultralytics
!pip install roboflow
!pip install dagshub
!pip install mlflow

In [3]:
from roboflow import Roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6842.2/8062.4 GB disk)


In [4]:
import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("dagshub")
from ultralytics import settings

dagshub.auth.add_app_token(token)
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Ship-Detection-in-SAR-Images')

# Optional: Set the experiment name
mlflow.set_experiment("tunning")


settings.update({'mlflow': True})

Accessing as AgabaEmbedded

Initialized MLflow to track repo "AgabaEmbedded/Ship-Detection-in-SAR-Images"

Repository AgabaEmbedded/Ship-Detection-in-SAR-Images initialized!

In [5]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="9VCfGRMmzvi61B4c9qu5")
project = rf.workspace("agabaembedded").project("ship-detection-sar-full")
version = project.version(11)
dataset = version.download("yolo26")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Ship-Detection-SAR-Full-11 in yolo26:: 100%|██████████| 23071/23071 [00:03<00:00, 6744.35it/s] 


In [6]:
!yolo train \
model= yolo26m.pt \
name= full-train-mega\
data=/kaggle/working/Ship-Detection-SAR-Full-11/data.yaml \
epochs=300 \
imgsz=640 \
device=[0, 1] \
plots=True \
pretrained = False\
save = True \
time= 11 \
deterministic=False \
optimizer = SGD \
cos_lr=True \
lr0=0.00683 \
lrf=0.00933 \
momentum=0.88752 \
weight_decay=0.00049 \
warmup_epochs=4.35648 \
warmup_momentum=0.55489 \
box= 6 \
cls=0.83514 \
dfl=1.20718 \
hsv_h=0.0 \
hsv_s=0.0 \
hsv_v=0.0 \
degrees=0.0 \
translate=0.0 \
scale=0.2 \
shear=0.0 \
perspective=0.0 \
flipud=0.0 \
fliplr=0.0 \
bgr=0.0 \
mosaic=0.0 \
mixup=0.0 \
cutmix=0.0 \
copy_paste=0.0

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=6, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.83514, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/Ship-Detection-SAR-Full-11/data.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.20718, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00683, lrf=0.00933, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.88752, mosaic=0.0, multi_

In [7]:
!yolo val model=/kaggle/working/runs/detect/full-train-mega/weights/best.pt data=/kaggle/working/Ship-Detection-SAR-Full-11/data.yaml split=val imgsz=640 batch=16

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1353.2±496.4 MB/s, size: 52.8 KB)
val: Scanning /kaggle/working/Ship-Detection-SAR-Full-11/valid/labels.cache... 692 images, 158 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 692/692 111.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 1.7it/s 25.8s
                   all        692       1027      0.906      0.888        0.9      0.403
Speed: 1.3ms preprocess, 33.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
💡 Learn more at https://docs.ultralytics.com/modes/val


In [8]:
!yolo val model=/kaggle/working/runs/detect/full-train-mega/weights/last.pt data=/kaggle/working/Ship-Detection-SAR-Full-11/data.yaml split=val imgsz=640 batch=16

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1399.0±450.0 MB/s, size: 50.4 KB)
val: Scanning /kaggle/working/Ship-Detection-SAR-Full-11/valid/labels.cache... 692 images, 158 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 692/692 107.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 1.7it/s 26.3s
                   all        692       1027      0.905      0.886      0.882      0.394
Speed: 1.2ms preprocess, 34.2ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2
💡 Learn more at https://docs.ultralytics.com/modes/val


In [9]:
!yolo val model=/kaggle/working/runs/detect/full-train-mega/weights/best.pt data=/kaggle/working/Ship-Detection-SAR-Full-11/data.yaml split=test imgsz=640 batch=16

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1395.1±610.7 MB/s, size: 74.5 KB)
val: Scanning /kaggle/working/Ship-Detection-SAR-Full-11/test/labels... 461 images, 103 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 461/461 1.2Kit/s 0.4s
val: New cache created: /kaggle/working/Ship-Detection-SAR-Full-11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.7it/s 17.1s
                   all        461        641      0.914      0.913      0.916      0.428
Speed: 1.5ms preprocess, 32.8ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val3
💡 Learn more at https://docs.ultralytics.com/modes/val


In [10]:
!yolo val model=/kaggle/working/runs/detect/full-train-mega/weights/last.pt data=/kaggle/working/Ship-Detection-SAR-Full-11/data.yaml split=test imgsz=640 batch=16

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1416.0±766.1 MB/s, size: 59.9 KB)
val: Scanning /kaggle/working/Ship-Detection-SAR-Full-11/test/labels.cache... 461 images, 103 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 461/461 56.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.6it/s 17.9s
                   all        461        641      0.908      0.903       0.91       0.42
Speed: 1.7ms preprocess, 34.4ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val4
💡 Learn more at https://docs.ultralytics.com/modes/val


In [11]:
!yolo export model=/kaggle/working/yolo26m.pt format=torchscript

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26m summary (fused): 132 layers, 20,411,132 parameters, 0 gradients, 68.2 GFLOPs

PyTorch: starting from '/kaggle/working/yolo26m.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (42.2 MB)

TorchScript: starting export with torch 2.10.0+cu128...
TorchScript: export success ✅ 5.8s, saved as '/kaggle/working/yolo26m.torchscript' (78.5 MB)

Export complete (7.8s)
Results saved to /kaggle/working
Predict:         yolo predict task=detect model=/kaggle/working/yolo26m.torchscript imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/yolo26m.torchscript imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
💡 Learn more at https://docs.ultralytics.com/modes/expor